In [2]:
# se importan las librerias necesarias 
import oracledb
import pandas as pd
import hashlib
import numpy as np
import re
from IPython.display import display
import warnings  # <--- 1. Importamos la librería de advertencias

# <--- 2. Le decimos a Python que ignore el Warning de Pandas
warnings.filterwarnings('ignore')


In [3]:
#se realiza la conexion a la base de datos 
usuario = "system" # Cambia por tu usuario de Oracle
contrasena = "Dulcee16$" 
dsn_tns = "localhost/xe" # o "localhost:1521/XE"

try: 
    conexion = oracledb.connect(user=usuario, password=contrasena, dsn=dsn_tns)
    print("¡Conexión exitosa a la base de datos Oracle (DataFlow Inc.)!")
except Exception as e:
    print(f"Error al conectar: {e}")

¡Conexión exitosa a la base de datos Oracle (DataFlow Inc.)!


In [4]:
#se crea la dataframe donde se realzara la limpieza 

df_cliente = df_clientes = pd.read_sql("SELECT * FROM CLIENTES", con=conexion)
print("--- CLIENTES CON DATOS FALTANTES (Nulos) ---")

# df.isnull().any(axis=1) filtra las filas que tienen al menos un NaN en cualquier columna 

clientes_nulos = df_clientes[df_clientes.isnull().any(axis=1)]
display(clientes_nulos)


--- CLIENTES CON DATOS FALTANTES (Nulos) ---


,IDCLIENTE,RUT,NOMBRE,APELLIDOPATERNO,APELLIDOMATERNO,TELEFONO,EMAIL,DIRECCION,CIUDAD,CODIGOPOSTAL,FECHAREGISTRO
0,21037,86.774.486-3,Pedro,None,None,+56970285742,uivrimmo939@demo.net,None,Puente Alto,8801,09-01-2018
1,21038,38.621.335-1,"Samuel, Zapata",None,None,+56920251843,qkppgfgo949@testmail.org,Avenida Esmeralda 6034,Colina,4803,25/02/2009
2,21039,77.954.6179,"Carmen, Zamora, Velasco",None,None,+56914486957,azxiscb464@example.com,Calle Rivadavia 6582 dpto. 139,Cauquenes,8255,05-26-2021
3,21040,None,Berta,Alvarado,Ruiz,+56916153741,wlmtehed493@testmail.org,Avenida Mitre 4014,Buin,6730,01-24-2023
4,21041,17065401-4,Salvador,Ortiz,None,+56958286572,ctkdt887@sample.cl,None,Ovalle,1839,2016-07-24
...,...,...,...,...,...,...,...,...,...,...,...
4994,25569,99.304.571-K,"Esteban, Arellano, Padilla",None,None,+56975630510,zgcpxc993@sample.cl,Camino La Paz 8928,La Serena,1617,29/04/2009
4996,25571,25666348-K,Gerardo,Miranda,None,+56925739813,None,Calle Sucre 7808,San Felipe,2078,01092020
4997,25572,80.105.9905,Oscar,None,None,None,ucvoldgy316@sample.cl,Calle Yrigoyen 6271 dpto. 1185,La Pintana,7425,2010-11-20
4998,25573,99657518-4,"Gabriela, Soto",None,None,+56945414243,dqvhcny980@example.com,Ruta Belgrano 3514,Osorno,7830,05102009


In [5]:
# PASO 3: LIMPIEZA DE DATOS, REGLAS DE NEGOCIO Y ÉTICA

# 1. Creamos una copia para no alterar el DataFrame original extraído
df_procesado = df_clientes.copy()

print("--- INICIANDO LIMPIEZA Y CIFRADO ÉTICO ---")

# FUNCIONES AUXILIARES

def cifrar_sensible(dato):
    """Aplica Hash SHA-256 para cumplir con la ética de privacidad"""
    if pd.isna(dato) or str(dato).strip() == "": 
        return None
    return hashlib.sha256(str(dato).encode('utf-8')).hexdigest()

def limpiar_rut(rut):
    """Deja el RUT solo con números y K, y le agrega el guion correcto"""
    if pd.isna(rut): 
        return None
    # Quitar puntos, guiones y cualquier carácter extraño
    limpio = re.sub(r'[^0-9Kk]', '', str(rut).upper())
    if len(limpio) > 1:
        return f"{limpio[:-1]}-{limpio[-1]}"
    return limpio


# APLICACIÓN DE REGLAS DE NEGOCIO (Estilo Pandas)


# REGLA 1: Descartar registros inválidos (Sin RUT)
df_procesado = df_procesado.dropna(subset=['RUT'])

# REGLA 2: Limpiar y estandarizar RUT (usando .apply igual que el profesor)
df_procesado['RUT_ESTANDAR'] = df_procesado['RUT'].apply(limpiar_rut)

# REGLA 3: Separar nombres que vienen en formato CSV (Ej: "Samuel, Zapata")
# Rellenamos los nulos temporalmente con texto vacío para que no falle el split
df_procesado['NOMBRE_RAW'] = df_procesado['NOMBRE'].fillna("")
nombres_split = df_procesado['NOMBRE_RAW'].str.split(',', expand=True)

# Asignamos las nuevas columnas dinámicamente
df_procesado['NUEVO_NOMBRE'] = nombres_split[0].str.strip()
df_procesado['NUEVO_APE_PAT'] = nombres_split[1].str.strip() if 1 in nombres_split.columns else df_procesado['APELLIDOPATERNO']
df_procesado['NUEVO_APE_MAT'] = nombres_split[2].str.strip() if 2 in nombres_split.columns else df_procesado['APELLIDOMATERNO']

# REGLA 4: Estandarizar Fechas a ISO 8601 (YYYY-MM-DD)
# Pandas maneja todos los formatos extraños automáticamente con to_datetime
df_procesado['FECHA_ESTANDAR'] = pd.to_datetime(df_procesado['FECHAREGISTRO'], errors='coerce', dayfirst=True).dt.strftime('%Y-%m-%d')

# REGLA 5: Manejar nulos en campos no críticos (como Dirección)
df_procesado['DIRECCION'] = df_procesado['DIRECCION'].fillna("NO DISPONIBLE")


# APLICACIÓN DE REGLAS ÉTICAS (DATOS SENSIBLES)

print("Aplicando cifrado criptográfico a Teléfonos y Correos...")

# Transformamos columnas completas aplicando la función de cifrado
df_procesado['TELEFONO_CIFRADO'] = df_procesado['TELEFONO'].apply(cifrar_sensible)
df_procesado['EMAIL_CIFRADO'] = df_procesado['EMAIL'].apply(cifrar_sensible)


# COMPROBACIÓN VISUAL (Como en 'Actividad 27-4')

print("\n--- COMPROBACIÓN: MUESTRA DE DATOS LIMPIOS Y CIFRADOS ---")

# Seleccionamos solo las columnas finales que nos interesan para ver el cambio
columnas_finales = [
    'RUT_ESTANDAR', 'NUEVO_NOMBRE', 'NUEVO_APE_PAT', 
    'FECHA_ESTANDAR', 'TELEFONO_CIFRADO', 'EMAIL_CIFRADO'
]

display(df_procesado[columnas_finales].head())


--- INICIANDO LIMPIEZA Y CIFRADO ÉTICO ---
Aplicando cifrado criptográfico a Teléfonos y Correos...

--- COMPROBACIÓN: MUESTRA DE DATOS LIMPIOS Y CIFRADOS ---


,RUT_ESTANDAR,NUEVO_NOMBRE,NUEVO_APE_PAT,FECHA_ESTANDAR,TELEFONO_CIFRADO,EMAIL_CIFRADO
0,86774486-3,Pedro,None,2018-01-09,5fe8ed5e154f5a1e2a302c8e22b24d3722489b1aaf6226...,8d26b464c789cab65471e8fcdd04e9228e78a940f5e1fb...
1,38621335-1,Samuel,Zapata,NaN,9f75d853a2bb65ffbfca575b8e619f6bced6bea6e1b1dd...,45dd8ee004e6643129663f26e7d41438236f0161f885be...
2,77954617-9,Carmen,Zamora,NaN,e8fab695236d589ff33618610eac241cf2b6036f030050...,0cce03be52d3b46c034a226b1f3d61c2c41e392a766932...
4,17065401-4,Salvador,None,NaN,40d407574666d59ff3b5fde5295b4d002e4d9a327e2328...,54ddd93e4d4565195679dfc3f939c4af6623b8bae3069f...
6,83854417-7,Cristina,Sandoval,NaN,ba1707469dc006ec624c7f755d827903dfde40dc009fab...,c2c5d1da1aa97ab76537264a85311fd467f4726cb05e01...


In [9]:

# PASO 3: LIMPIEZA PROFESIONAL 

# Creamos la copia de trabajo
cliente_procesado = df_clientes.copy()


# 1. TRATAMIENTO PREVENTIVO DE NULOS

# Reemplazamos los NaN por strings vacíos o etiquetas profesionales 
# para que las funciones de limpieza no se encuentren con valores nulos.

cliente_procesado['RUT'] = cliente_procesado['RUT'].fillna('SIN RUT')
cliente_procesado['NOMBRE'] = cliente_procesado['NOMBRE'].fillna('Nombre No Registrado')
cliente_procesado['APELLIDOPATERNO'] = cliente_procesado['APELLIDOPATERNO'].fillna('Apellido No Registrado')
cliente_procesado['APELLIDOMATERNO'] = cliente_procesado['APELLIDOMATERNO'].fillna('Apellido No Registrado')
cliente_procesado['TELEFONO'] = cliente_procesado['TELEFONO'].fillna('Sin Contacto')
cliente_procesado['EMAIL'] = cliente_procesado['EMAIL'].fillna('Sin Email')
cliente_procesado['DIRECCION'] = cliente_procesado['DIRECCION'].fillna('Dirección No Proporcionada')
cliente_procesado['CIUDAD'] = cliente_procesado['CIUDAD'].fillna('Sin Ciudad')

# ------------------------------------------------------------
# 2. FUNCIONES DE TRANSFORMACIÓN MEJORADAS
# ------------------------------------------------------------

def estandarizar_rut_pro(rut):
    if rut == 'SIN RUT': return 'RUT NO REGISTRADO'
    limpio = re.sub(r'[^0-9Kk]', '', str(rut).upper())
    return f"{limpio[:-1]}-{limpio[-1]}" if len(limpio) > 1 else 'RUT INVÁLIDO'

def cifrar_dato_pro(dato):
    # Si el dato es una de nuestras etiquetas de "Sin registro", no lo ciframos
    etiquetas_vacias = ['Sin Contacto', 'Sin Email', 'Nombre No Registrado']
    if dato in etiquetas_vacias:
        return dato
    return hashlib.sha256(str(dato).encode('utf-8')).hexdigest()

# ------------------------------------------------------------
# 3. APLICACIÓN DE REGLAS DE NEGOCIO
# ------------------------------------------------------------

# A) RUT
cliente_procesado['RUT_ESTANDAR'] = cliente_procesado['RUT'].apply(estandarizar_rut_pro)

# B) Nombres y Apellidos (Manejo de CSV y NaN)
# Separamos la columna NOMBRE que puede traer "Samuel, Zapata"
nombres_split = cliente_procesado['NOMBRE'].str.split(',', expand=True)

# 3. El primer pedazo SIEMPRE es el Nombre
cliente_procesado['NUEVO_NOMBRE'] = nombres_split[0].str.strip()
#Si el split dio 'None' (porque no había coma), usamos fillna para traer el dato de la columna original.
if 1 in nombres_split.columns:
    cliente_procesado['NUEVO_APE_PAT'] = nombres_split[1].str.strip().fillna(cliente_procesado['APELLIDOPATERNO'])
else:
    cliente_procesado['NUEVO_APE_PAT'] = cliente_procesado['APELLIDOPATERNO']

# 5. Apellido Materno: Misma lógica con el pedazo 3 del split.
if 2 in nombres_split.columns:
    cliente_procesado['NUEVO_APE_MAT'] = nombres_split[2].str.strip().fillna(cliente_procesado['APELLIDOMATERNO'])
else:
    cliente_procesado['NUEVO_APE_MAT'] = cliente_procesado['APELLIDOMATERNO']

# C) Fechas (Crucial: evitar NaN en fecha)
# Convertimos a fecha, los errores se vuelven NaT (Not a Time)
fechas_temp = pd.to_datetime(cliente_procesado['FECHAREGISTRO'], errors='coerce', dayfirst=True)
# Reemplazamos NaT por una fecha base profesional (1900-01-01)
cliente_procesado['FECHA_ESTANDAR'] = fechas_temp.dt.strftime('%Y-%m-%d').fillna('1900-01-01')

# D) Cifrado Ético sin NaN
cliente_procesado['TELEFONO_CIFRADO'] = cliente_procesado['TELEFONO'].apply(cifrar_dato_pro)
cliente_procesado['EMAIL_CIFRADO'] = cliente_procesado['EMAIL'].apply(cifrar_dato_pro)

# ------------------------------------------------------------
# 4. REVISIÓN FINAL (Comprobación de 0 nulos)
# ------------------------------------------------------------
columnas_finales = [
    'RUT_ESTANDAR', 'NUEVO_NOMBRE', 'NUEVO_APE_PAT', 'NUEVO_APE_MAT',
    'FECHA_ESTANDAR', 'TELEFONO_CIFRADO', 'EMAIL_CIFRADO', 'DIRECCION'
]

print("--- COMPROBACIÓN FINAL DE NULOS (Debe ser 0 en todo) ---")
print(cliente_procesado[columnas_finales].isnull().sum())

print("\n--- MUESTRA PROFESIONAL SIN NaN ---")
display(cliente_procesado[columnas_finales].head(10))

--- COMPROBACIÓN FINAL DE NULOS (Debe ser 0 en todo) ---
RUT_ESTANDAR        0
NUEVO_NOMBRE        0
NUEVO_APE_PAT       0
NUEVO_APE_MAT       0
FECHA_ESTANDAR      0
TELEFONO_CIFRADO    0
EMAIL_CIFRADO       0
DIRECCION           0
dtype: int64

--- MUESTRA PROFESIONAL SIN NaN ---


,RUT_ESTANDAR,NUEVO_NOMBRE,NUEVO_APE_PAT,NUEVO_APE_MAT,FECHA_ESTANDAR,TELEFONO_CIFRADO,EMAIL_CIFRADO,DIRECCION
0,86774486-3,Pedro,Apellido No Registrado,Apellido No Registrado,2018-01-09,5fe8ed5e154f5a1e2a302c8e22b24d3722489b1aaf6226...,8d26b464c789cab65471e8fcdd04e9228e78a940f5e1fb...,Dirección No Proporcionada
1,38621335-1,Samuel,Zapata,Apellido No Registrado,1900-01-01,9f75d853a2bb65ffbfca575b8e619f6bced6bea6e1b1dd...,45dd8ee004e6643129663f26e7d41438236f0161f885be...,Avenida Esmeralda 6034
2,77954617-9,Carmen,Zamora,Velasco,1900-01-01,e8fab695236d589ff33618610eac241cf2b6036f030050...,0cce03be52d3b46c034a226b1f3d61c2c41e392a766932...,Calle Rivadavia 6582 dpto. 139
3,RUT NO REGISTRADO,Berta,Alvarado,Ruiz,1900-01-01,c42ae5329508269e49d293fcb7e28babad695b88f50ac0...,9274420a283041aa9a249aef4737e6a395932c61b31506...,Avenida Mitre 4014
4,17065401-4,Salvador,Ortiz,Apellido No Registrado,1900-01-01,40d407574666d59ff3b5fde5295b4d002e4d9a327e2328...,54ddd93e4d4565195679dfc3f939c4af6623b8bae3069f...,Dirección No Proporcionada
5,RUT NO REGISTRADO,Julio,Apellido No Registrado,Apellido No Registrado,1900-01-01,c18ebda7c0f9405633ac7d5093763e24274cedf25e3199...,Sin Email,Ruta Independencia 4271 dpto. 147
6,83854417-7,Cristina,Sandoval,Apellido No Registrado,1900-01-01,ba1707469dc006ec624c7f755d827903dfde40dc009fab...,c2c5d1da1aa97ab76537264a85311fd467f4726cb05e01...,Avenida San Mart�n 9219 dpto. 1557
7,85020977-8,Natalia,Luna,Luna,1900-01-01,977f434eac85a80e0cd6f1d4b11136daa69aa100f2c212...,Sin Email,Ruta Sarmiento 5370
8,RUT NO REGISTRADO,�lex,Delgado,Delgado,1900-01-01,16cca24b3a19cd04034dc27e47f030861e5cfa1b496949...,Sin Email,Camino Rivadavia 7034
9,24282246-3,Agust�n,L�pez,Apellido No Registrado,1900-01-01,Sin Contacto,12a54d5ee7dcef1280d53dd11e7ec5d405223c9b678b6e...,Avenida Yrigoyen 8230 dpto. 1373
